# Governança — Mascaramento de PII (Unity Catalog)

Cria as **column masks** do Unity Catalog para as colunas sensíveis das dimensões
(`dim_clientes`, `dim_assistentes`) e as aplica. O valor em claro fica disponível
apenas para o grupo privilegiado (Entra ID). Usa a lib `security`.

Execute **uma vez** após a criação das dimensões na Bronze.

In [ ]:
# ===================== PARÂMETROS (Widgets) =====================
import sys

sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("bronze_schema", "b_dm_callcenter")
dbutils.widgets.text("privileged_group", "dm_pii_readers")

CATALOG    = dbutils.widgets.get("catalog")
SCHEMA     = dbutils.widgets.get("bronze_schema")
PRIV_GROUP = dbutils.widgets.get("privileged_group")

from security import column_mask_functions_sql, apply_column_masks_sql, PII_COLUMNS

print("Catálogo:", CATALOG, "| Schema:", SCHEMA, "| Grupo PII:", PRIV_GROUP)

## Helper — executa um script SQL com múltiplos statements

In [ ]:
def run_sql_script(sql_text: str):
    """Executa cada statement (separado por ';') individualmente."""
    for stmt in sql_text.split(";"):
        s = stmt.strip()
        if s:
            spark.sql(s)

## 1) Cria as funções de máscara (reutilizáveis no schema)

In [ ]:
sql_funcs = column_mask_functions_sql(CATALOG, SCHEMA, privileged_group=PRIV_GROUP)
print(sql_funcs)
run_sql_script(sql_funcs)
print("[OK] Funções de máscara criadas.")

## 2) Aplica as máscaras às colunas sensíveis das dimensões

In [ ]:
for table, cols in PII_COLUMNS.items():
    sql_apply = apply_column_masks_sql(CATALOG, SCHEMA, table, cols.keys())
    if sql_apply:
        print(sql_apply)
        run_sql_script(sql_apply)
        print(f"[OK] Máscaras aplicadas em {CATALOG}.{SCHEMA}.{table}")

## 3) Validação
Confirme as máscaras aplicadas nas colunas.

In [ ]:
for table in PII_COLUMNS:
    print(f"--- {CATALOG}.{SCHEMA}.{table} ---")
    spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{SCHEMA}.{table}").display()